# 00 — MPS Playground

Guided tour of the Apple-GPU (`mps`) backend before we build anything real. This is exploration,
not training — per project rules, real training runs live in `scripts/`, not notebooks.

Topics: tensors on `mps`, autocast dtypes (why bf16), the timing-without-`synchronize()` trap,
and reading GPU memory. Use the **`llm-lab`** kernel (registered by `scripts/setup.sh`).

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import time
import torch

import sys
sys.path.insert(0, "../src")
from llmlab.utils import get_device, mem_stats, set_seed

set_seed(42)
device = get_device()
print(f"torch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"Using device: {device}")

## 1. Tensors on MPS

Same API as CUDA: `.to(device)` moves data; ops run on the GPU transparently. The gotcha is
that MPS uses Apple Silicon's **unified memory** — there's no separate VRAM, the GPU and CPU
share the same physical RAM, which is part of why our 16GB is a real constraint everywhere,
not just for the model.

In [ ]:
x = torch.randn(4, 4, device=device)
y = torch.randn(4, 4, device=device)
z = x @ y
print(z)
print(f"z.device = {z.device}, z.dtype = {z.dtype}")

## 2. Autocast dtypes: why bf16, not fp16

`fp16` (1 sign / 5 exponent / 10 mantissa bits) has a narrow dynamic range — large activations
or gradients overflow to `inf`/`nan` easily, historically needing loss scaling to compensate.
`bf16` (1 / 8 / 7 bits) has the **same exponent range as fp32** but less precision (mantissa).
For transformer training, range matters more than precision, so bf16 lets us mix precision
**without loss scaling** — simpler and more numerically stable. This project uses bf16
everywhere on MPS (fp16 is known to NaN in softmax on this backend).

In [ ]:
a = torch.randn(512, 512, device=device)
b = torch.randn(512, 512, device=device)

with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
    out_bf16 = a @ b
print(f"autocast bf16 -> {out_bf16.dtype}")

out_fp32 = a @ b
print(f"no autocast   -> {out_fp32.dtype}")

# Range check: bf16 keeps fp32's exponent range, just loses mantissa precision.
big = torch.tensor([1e30], dtype=torch.float32)
print(f"1e30 as bf16: {big.to(torch.bfloat16).item():.3e} (survives)")
print(f"1e30 as fp16: {big.to(torch.float16).item()} (overflows to inf)")

## 3. Timing pitfall: MPS ops are async — you MUST synchronize before timing

GPU kernels are launched asynchronously; `time.perf_counter()` around an op only measures how
long it took to **queue** the work, not to finish it, unless you force a sync point. Compare
the two cells below — the "wrong" one will report a suspiciously tiny, batch-size-independent
time.

In [ ]:
n = 4096
a = torch.randn(n, n, device=device)
b = torch.randn(n, n, device=device)

# warm up (first call pays for kernel compilation)
for _ in range(3):
    _ = a @ b
torch.mps.synchronize()

# WRONG: no synchronize -> just measures dispatch/queueing time
start = time.perf_counter()
c = a @ b
wrong_elapsed = time.perf_counter() - start
print(f"without sync: {wrong_elapsed * 1000:.3f} ms  <- misleadingly fast")

# RIGHT: synchronize forces the CPU to wait for the GPU to actually finish
start = time.perf_counter()
c = a @ b
torch.mps.synchronize()
right_elapsed = time.perf_counter() - start
print(f"with sync:    {right_elapsed * 1000:.3f} ms  <- the real number")

## 4. Memory readout

`mem_stats()` reports process RSS (`psutil`) alongside MPS-specific allocator stats. On MPS
there's no `nvidia-smi` equivalent, so `torch.mps.current_allocated_memory()` /
`driver_allocated_memory()` are the closest things we have to a memory profiler.

In [ ]:
print("before allocation:", mem_stats())

big_tensor = torch.randn(4096, 4096, device=device)
print("after allocating a 4096x4096 tensor:", mem_stats())

del big_tensor
torch.mps.empty_cache()
print("after del + empty_cache:", mem_stats())

## 5. What `scripts/bench_mps.py` found (see D-008 in `docs/DECISIONS.md`)

Headline results from the full benchmark script (run from terminal, not here — it's a
long-running measurement, not exploration):

- bf16 matmul: ~3.5-3.8 TFLOPS (1k-4k square matrices); fp32 is somewhat slower.
- A throwaway ~9.1M-param GPT hits ~20-24k fwd+bwd tokens/sec depending on `seq_len`.
- **The interesting finding:** throughput doesn't degrade gracefully as micro-batch grows —
  it's flat, then falls off a cliff (3-15x slower) at a specific allocation size, well before
  system RAM or MPS's own "recommended max memory" would suggest trouble, and well before a
  hard OOM error. Lesson: **pick micro-batch from the measured plateau, not from how much you
  can technically allocate.**
- Extrapolating those numbers to the 100M-param hero run flags a real tension with the
  original plan (D-001 assumed 1-3 days; the math here suggests more like 1.5-3 weeks for a
  Chinchilla-sized token budget). Worth revisiting before phase 9.